In [1]:
import numpy as np
import pandas as pd
from itertools import product
from collections import Counter, defaultdict
from tqdm import tqdm

In [2]:
def get_candidate_thresholds(values, max_thresholds: int = 20):
    if not np.issubdtype(values.dtype, np.number):
        return []

    mask = ~np.isnan(values)
    unique_vals = np.unique(values[mask])

    if len(unique_vals) <= max_thresholds:
        thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2
    else:
        quantiles = np.linspace(0, 1, max_thresholds + 1)
        thresholds = np.quantile(unique_vals, quantiles)[1:-1]

    return thresholds

In [3]:
class DecisionTreeRegressorCustom:
    def __init__(
        self,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        cat_features: list[str] | None = None,
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.cat_features = cat_features if cat_features else []
        self.tree = None

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.features = list(X.columns)
        self.cat_features = [f for f in self.features if f in self.cat_features]
        self.n_features = len(self.features)

        self.X = X.values    
        self.y = y.values
        self.feature_to_idx = {f: i for i, f in enumerate(self.features)}

        idxs = np.arange(len(y))
        self.tree = self._build_tree(idxs, depth=0)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        X_np = X.values
        preds = np.empty(X.shape[0])
        for i in range(X.shape[0]):
            preds[i] = self._predict_row(X_np[i], self.tree)
        return preds

    def _build_tree(self, idxs, depth: int):
        y = self.y[idxs]
        num_samples = len(idxs)

        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or num_samples < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return {"type": "leaf", "value": y.mean()}

        best_split = self._find_best_split(idxs)
        if best_split is None:
            return {"type": "leaf", "value": y.mean()}

        feature, threshold, left_idxs, right_idxs = best_split
        if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
            return {"type": "leaf", "value": y.mean()}

        return {
            "type": "node",
            "feature": feature,
            "threshold": threshold,
            "left": self._build_tree(left_idxs, depth + 1),
            "right": self._build_tree(right_idxs, depth + 1),
        }

    def _find_best_split(self, idxs):
        best_feature, best_threshold, best_mse = None, None, float("inf")
        best_left = best_right = None
        y = self.y[idxs]

        n_feats = self.n_features
        max_feats = int(n_feats) 
        features_subset = np.random.choice(self.features, max_feats, replace=False)

        for feature in features_subset:
            f_idx = self.feature_to_idx[feature]
            values = self.X[idxs, f_idx]

            if feature in self.cat_features:
                cats = np.unique(values[~pd.isna(values)])
                if len(cats) <= 1:
                    continue
                for cat in cats:
                    left_mask = values == cat
                    right_mask = ~left_mask
                    if (
                        left_mask.sum() < self.min_samples_leaf
                        or right_mask.sum() < self.min_samples_leaf
                    ):
                        continue
                    mse = self._calc_mse_split(y, left_mask, right_mask)
                    if mse < best_mse:
                        best_feature, best_threshold, best_mse = feature, cat, mse
                        best_left = idxs[left_mask]
                        best_right = idxs[right_mask]

            else:
                thresholds = get_candidate_thresholds(values)
                for thr in thresholds:
                    left_mask = values <= thr
                    right_mask = values > thr
                    if (
                        left_mask.sum() < self.min_samples_leaf
                        or right_mask.sum() < self.min_samples_leaf
                    ):
                        continue
                    mse = self._calc_mse_split(y, left_mask, right_mask)
                    if mse < best_mse:
                        best_feature, best_threshold, best_mse = feature, thr, mse
                        best_left = idxs[left_mask]
                        best_right = idxs[right_mask]

        if best_feature is None:
            return None
        return best_feature, best_threshold, best_left, best_right

    def _calc_mse_split(self, y, left_mask, right_mask):
        left_y, right_y = y[left_mask], y[right_mask]
        total = len(left_y) + len(right_y)
        mse = (
            len(left_y) * np.var(left_y, ddof=0) + len(right_y) * np.var(right_y, ddof=0)
        ) / total
        return mse

    def _predict_row(self, row, node):
        if node["type"] == "leaf":
            return node["value"]

        feature, threshold = node["feature"], node["threshold"]
        f_idx = self.feature_to_idx[feature]
        val = row[f_idx]

        if feature in self.cat_features:            
            if pd.isna(val):
                return (self._predict_row(row, node["left"]) + self._predict_row(row, node["right"])) / 2
            return (
                self._predict_row(row, node["left"])
                if val == threshold
                else self._predict_row(row, node["right"])
            )
        else:                                        
            if pd.isna(val):
                return (self._predict_row(row, node["left"]) + self._predict_row(row, node["right"])) / 2
            return (
                self._predict_row(row, node["left"])
                if val <= threshold
                else self._predict_row(row, node["right"])
            )

In [4]:
class RandomForestRegressorCustom:
    def __init__(
        self,
        n_estimators: int = 100,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        cat_features: list[str] | None = None,
        random_state: int | None = None,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.cat_features = cat_features if cat_features else []
        self.random_state = random_state
        self.trees: list[DecisionTreeRegressorCustom] = []
        self.bootstraps_idx: list[np.ndarray] = []

    def fit(self, X: pd.DataFrame, y: pd.Series):
        np.random.seed(self.random_state)
        self.X, self.y = X.reset_index(drop=True), y.reset_index(drop=True)
        self.trees, self.bootstraps_idx = [], []

        for _ in tqdm(range(self.n_estimators), desc="Training trees"):
            idxs = np.random.choice(len(self.X), size=len(self.X), replace=True)
            self.bootstraps_idx.append(idxs)

            tree = DecisionTreeRegressorCustom(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                cat_features=self.cat_features,
            )
            tree.fit(self.X.iloc[idxs], self.y.iloc[idxs])
            self.trees.append(tree)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        preds = np.zeros(len(X))
        for tree in self.trees:
            preds += tree.predict(X)
        return preds / self.n_estimators

    def oob_score(self) -> float:
        oob_preds = np.zeros(len(self.X))
        oob_counts = np.zeros(len(self.X))

        for tree, idxs in zip(self.trees, self.bootstraps_idx):
            oob_idx = np.setdiff1d(np.arange(len(self.X)), idxs)
            if len(oob_idx) == 0:
                continue
            oob_preds[oob_idx] += tree.predict(self.X.iloc[oob_idx])
            oob_counts[oob_idx] += 1

        mask = oob_counts > 0
        oob_preds[mask] /= oob_counts[mask]
        mse = np.mean((self.y[mask] - oob_preds[mask]) ** 2)
        return mse

In [5]:
def preprocess_data(df: pd.DataFrame, cat_features: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if col in cat_features:
            mode_val = df[col].mode()
            df[col] = df[col].fillna(mode_val[0] if len(mode_val) else "missing")
        else:
            df[col] = df[col].fillna(df[col].median())
    return df

In [6]:
def grid_search_rf_oob(
    X: pd.DataFrame,
    y: pd.Series,
    cat_features: list[str],
    param_grid: dict,
    n_estimators: int = 30,
    random_state: int = 42,
):
    best_params, best_mse, results = None, float("inf"), []

    total = np.prod([len(v) for v in param_grid.values()])
    print(f"Total hyperparameter combinations: {total}")

    for params in tqdm(product(*param_grid.values()), total=total):
        param_dict = dict(zip(param_grid.keys(), params))

        model = RandomForestRegressorCustom(
            n_estimators=n_estimators,
            max_depth=param_dict["max_depth"],
            min_samples_split=param_dict["min_samples_split"],
            min_samples_leaf=param_dict["min_samples_leaf"],
            cat_features=cat_features,
            random_state=random_state,
        )
        model.fit(X, y)
        mse = model.oob_score()

        results.append((mse, param_dict))
        if mse < best_mse:
            best_mse, best_params = mse, param_dict

        print(f"Params: {param_dict}, OOB-MSE: {mse:.4f}")

    print(f"\nBest params: {best_params}  |  Best OOB-MSE: {best_mse:.4f}")
    return best_params, results

In [7]:
def parse_flexible_date(d):
    if pd.isna(d):
        return pd.NaT
    parts = d.split('-')
    try:
        if len(parts) == 1: 
            return pd.to_datetime(f"{parts[0]}-06-15")
        elif len(parts) == 2:  
            return pd.to_datetime(f"{parts[0]}-{parts[1]}-15")
        elif len(parts) == 3:
            return pd.to_datetime(d)
        else:
            return pd.NaT
    except:
        return pd.NaT

In [8]:
df = pd.read_csv("/kaggle/input/sporify-songs/train.csv")

y = df["popularity"]
X = df.drop(columns=["popularity", "Unnamed: 0"])

columns_to_drop = ['track_href', 'uri',\
                   'analysis_url', 'track_id', 'track_name', 'track_album_id', 'id', 'playlist_id', 'type']
X=X.drop(columns=columns_to_drop)

mean_val = X['duration_ms'].mean()
X['duration_ms'] = X['duration_ms'].fillna(mean_val)

X['parsed_date'] = X['track_album_release_date'].apply(parse_flexible_date)

X['year'] = X['parsed_date'].dt.year
X['month'] = X['parsed_date'].dt.month
X['day'] = X['parsed_date'].dt.day
X['day_of_week'] = X['parsed_date'].dt.weekday  
X['is_weekend'] = X['day_of_week'].isin([5, 6]).astype(int)
X['day_of_year'] = X['parsed_date'].dt.dayofyear
X['quarter'] = X['parsed_date'].dt.quarter
X['decade'] = (X['year'] // 10) * 10
X['is_recent'] = (X['year'] >= 2008).astype(int)

bins = [0, 180000, 300000, float('inf')] 
labels = [0, 1, 2]
X['duration_category'] = pd.cut(X['duration_ms'], bins=bins, labels=labels, include_lowest=True).astype(int)

X['energy_dance_ratio'] = X['energy'] / (X['danceability'] + 1e-6)
X['mood_score'] = X['valence'] * X['energy']
X['acoustic_sad'] = X['acousticness'] * (1 - X['valence'])

X.drop(columns=['track_album_release_date', 'parsed_date'], inplace=True)

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
df_proc = preprocess_data(X, cat_features)

param_grid = {
    "max_depth": [30],
    "min_samples_split": [7, 10],
    "min_samples_leaf": [5, 7],
}

In [9]:
best_params, _ = grid_search_rf_oob(
    df_proc, y,
    cat_features,
    param_grid,
    n_estimators=30,
    random_state=42
)

Total hyperparameter combinations: 4


 25%|██▌       | 1/4 [02:28<07:25, 148.48s/it]

Params: {'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 5}, OOB-MSE: 133.2431



 50%|█████     | 2/4 [04:49<04:48, 144.03s/it]

Params: {'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 7}, OOB-MSE: 135.3076



 75%|███████▌  | 3/4 [07:23<02:28, 148.80s/it]

Params: {'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 5}, OOB-MSE: 133.8249



100%|██████████| 4/4 [09:45<00:00, 146.37s/it]

Params: {'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 7}, OOB-MSE: 134.7397

Best params: {'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 5}  |  Best OOB-MSE: 133.2431


In [10]:
rf_final = RandomForestRegressorCustom(
    n_estimators=30, 
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    cat_features=cat_features,
    random_state=42
)

rf_final.fit(df_proc, y)

Training trees: 100%|██████████| 30/30 [02:27<00:00,  4.91s/it]


In [11]:
df_test = pd.read_csv("/kaggle/input/sporify-songs/test.csv")
X_test = df_test.drop(columns=["Unnamed: 0"], errors="ignore")
X_test = X_test.drop(columns=columns_to_drop, errors="ignore")

X_test['duration_ms'] = X_test['duration_ms'].fillna(mean_val)

X_test['parsed_date'] = X_test['track_album_release_date'].apply(parse_flexible_date)
X_test['year']        = X_test['parsed_date'].dt.year
X_test['month']       = X_test['parsed_date'].dt.month
X_test['day']         = X_test['parsed_date'].dt.day
X_test['day_of_week'] = X_test['parsed_date'].dt.weekday
X_test['is_weekend']  = X_test['day_of_week'].isin([5,6]).astype(int)
X_test['day_of_year'] = X_test['parsed_date'].dt.dayofyear
X_test['quarter']     = X_test['parsed_date'].dt.quarter

X_test['decade'] = (X_test['year'] // 10) * 10
X_test['is_recent'] = (X_test['year'] >= 2008).astype(int)

bins   = [0, 180000, 300000, float('inf')]
labels = [0, 1, 2]
X_test['duration_category'] = (
    pd.cut(
        X_test['duration_ms'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    .astype(int)
)

X_test['energy_dance_ratio'] = X_test['energy'] / (X_test['danceability'] + 1e-6)
X_test['mood_score']         = X_test['valence'] * X_test['energy']
X_test['acoustic_sad']       = X_test['acousticness'] * (1 - X_test['valence'])

X_test.drop(columns=['track_album_release_date','parsed_date'], inplace=True)
cat_features = X.select_dtypes(include=["object","category"]).columns.tolist()
X_test_proc = preprocess_data(X_test, cat_features)

In [12]:
preds_test = rf_final.predict(X_test_proc)

In [13]:
submission = pd.DataFrame({
    'Id': np.arange(len(preds_test)),
    'popularity': preds_test 
})

submission.to_csv('submission.csv', index=False)

## Тут опишу несколько мыслей по поводу моей работы. 
### Вначале, как и в последней версии дерева, при поиске лучшего признака для сплита, для категориальных признаков перебираются уникальные категории и разбивают лист на left - объекты с этим значением (==cat) и right - все остальные (!=cat), это почему то оказалось лучше, чем упорядовачение категорий по таргету (среднее значение таргета для отдельной категории) и кодирование категорий согласно порядку и брать трешхолд как для числовых значений. Я не рассматривал вариант с перебором двух множеств для разбиения, так как нужно будет перебрать 2^(M-1)-1 возможных сплитов
### Помогла нарастить качество out_of_bag оценка, до этого я делил тренировочные данные на train_data и val_data, а потом для train_data еще делал подвыборку и бутстрапа, поэтому деревья у меня тренировались где то на 0.8*0.66 данных, с out_of_bag модель получала больше данных и обучалась лучше.
### Совсем не дало прироста, а даже наоборот сократило скор - подвыборка признаков для каждого листа из дерева, казалось бы, у нас деревья станут разнообразнее у скор будет лучше, но подвыборка из m/3 и даже 2**(m/3) давала намного хуже резульаты.
### Feature engineering почти не дал прироста, возможно плохо искал